# Task 4 — Integrated NLP Product Review Web Application

This deployment is built directly from the three project notebooks:

- **Task 1:** fine-tuned RoBERTa sentiment classifier
- **Task 2:** product clustering and six `kmeans_category` meta-categories
- **Task 3:** Qwen-generated recommendation articles

## Application tabs

1. **Sentiment Analyzer** — classify a new review as Negative, Neutral, or Positive with confidence scores.
2. **Category Explorer** — inspect products and review statistics from Task 2 clusters.
3. **AI Recommendations** — display the recommendation articles generated in Task 3.

### Files produced by the previous notebooks

```text
models/amazon_roberta_sentiment_model/
datasets/clustered_reviews.csv
category_recommendation_articles.csv
```

Run Tasks 1–3 first so these artifacts exist.

## 1. Install dependencies

In [ ]:
# !pip install -q -U gradio transformers torch pandas safetensors

## 2. Imports

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import gradio as gr

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

print("PyTorch:", torch.__version__)
print("Gradio:", gr.__version__)

PyTorch: 2.11.0+cu130
Gradio: 6.29.0


## 3. Project paths

In [25]:
MODEL_PATH = Path("model/amazon_roberta_sentiment_model")
CLUSTERED_REVIEWS_PATH = Path("datasets/clustered_reviews.csv")
ARTICLES_PATH = Path("category_recommendation_articles.csv")

required_paths = [
    MODEL_PATH,
    CLUSTERED_REVIEWS_PATH,
    ARTICLES_PATH
]

for path in required_paths:
    print(
        "FOUND" if path.exists() else "MISSING",
        "->",
        path
    )

FOUND -> model/amazon_roberta_sentiment_model
FOUND -> datasets/clustered_reviews.csv
FOUND -> category_recommendation_articles.csv


## 4. Load the Task 1 sentiment model

Task 1 saves the fine-tuned model to `models/amazon_roberta_sentiment_model`.

In [26]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        "Task 1 model not found. Run Task 1 and save the model first."
    )

sentiment_tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_PATH)
)

sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_PATH)
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

sentiment_model = sentiment_model.to(device)
sentiment_model.eval()

print("Device:", device)
print("Saved labels:", sentiment_model.config.id2label)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Device: cuda
Saved labels: {0: 'negative', 1: 'neutral', 2: 'positive'}


## 5. Safe Task 1 label mapping

In [27]:
DEFAULT_LABELS = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

def build_label_mapping(model):
    stored = getattr(
        model.config,
        "id2label",
        {}
    ) or {}

    mapping = {}

    for i in range(model.config.num_labels):
        label = stored.get(
            i,
            stored.get(str(i), "")
        )
        label = str(label)

        if (
            not label
            or label.upper().startswith("LABEL_")
        ):
            label = DEFAULT_LABELS.get(
                i,
                f"Class {i}"
            )
        else:
            label = label.capitalize()

        mapping[i] = label

    return mapping

ID2LABEL = build_label_mapping(
    sentiment_model
)

print(ID2LABEL)

{0: 'Negative', 1: 'Neutral', 2: 'Positive'}


## 6. Sentiment prediction function

In [28]:
def predict_sentiment(review):

    if review is None or not review.strip():
        return (
            "Please enter a review.",
            {}
        )

    inputs = sentiment_tokenizer(
        review,
        return_tensors="pt",
        truncation=True,
        max_length=256,
        padding=True
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        logits = sentiment_model(
            **inputs
        ).logits

        probabilities = torch.softmax(
            logits,
            dim=-1
        )[0]

    predicted_id = int(
        torch.argmax(
            probabilities
        ).item()
    )

    scores = {
        ID2LABEL[i]:
            float(probabilities[i].item())
        for i in range(len(probabilities))
    }

    scores = dict(
        sorted(
            scores.items(),
            key=lambda x: x[1],
            reverse=True
        )
    )

    return (
        ID2LABEL[predicted_id],
        scores
    )

# Task 2 Data

## 7. Load clustered reviews

Your Task 2 notebook exports `datasets/clustered_reviews.csv` with:

- `product_name`
- `kmeans_cluster`
- `kmeans_category`
- review text/rating and product metadata.

In [29]:
if not CLUSTERED_REVIEWS_PATH.exists():
    raise FileNotFoundError(
        "Task 2 output datasets/clustered_reviews.csv was not found."
    )

clustered = pd.read_csv(
    CLUSTERED_REVIEWS_PATH,
    low_memory=False
)

required_cluster_columns = [
    "product_name",
    "kmeans_cluster",
    "kmeans_category",
    "reviews.rating",
    "reviews.text"
]

missing = [
    col for col in required_cluster_columns
    if col not in clustered.columns
]

if missing:
    raise ValueError(
        f"Task 2 output is missing: {missing}"
    )

print("Reviews:", len(clustered))
print(
    "Categories:",
    clustered["kmeans_category"].nunique()
)

display(
    clustered[
        "kmeans_category"
    ].value_counts()
)

Reviews: 60841
Categories: 6


,count
kmeans_category,
Fire Tablets,29635
"Household, Office & Pets",12140
Smart Speakers & Echo,8737
Streaming & TV,5087
E-readers & Kindle,5064
Chargers & Cables,178


## 8. Prepare Task 2 category summary

In [30]:
category_summary = (
    clustered
    .groupby("kmeans_category")
    .agg(
        products=("product_name", "nunique"),
        reviews=("reviews.text", "count"),
        average_rating=("reviews.rating", "mean")
    )
    .reset_index()
    .sort_values(
        "reviews",
        ascending=False
    )
)

category_summary[
    "average_rating"
] = category_summary[
    "average_rating"
].round(2)

display(category_summary)

,kmeans_category,products,reviews,average_rating
2,Fire Tablets,24,29635,4.52
3,"Household, Office & Pets",14,12140,4.45
4,Smart Speakers & Echo,10,8737,4.66
5,Streaming & TV,6,5087,4.71
1,E-readers & Kindle,16,5063,4.70
0,Chargers & Cables,8,178,4.00


## 9. Category Explorer function

In [31]:
CATEGORY_CHOICES = sorted(
    clustered[
        "kmeans_category"
    ]
    .dropna()
    .unique()
    .tolist()
)

def explore_category(category):

    if not category:
        return (
            pd.DataFrame(),
            pd.DataFrame()
        )

    subset = clustered[
        clustered[
            "kmeans_category"
        ] == category
    ].copy()

    summary = (
        subset
        .groupby("product_name")
        .agg(
            review_count=(
                "reviews.rating",
                "count"
            ),
            average_rating=(
                "reviews.rating",
                "mean"
            )
        )
        .reset_index()
        .sort_values(
            [
                "average_rating",
                "review_count"
            ],
            ascending=[
                False,
                False
            ]
        )
    )

    summary[
        "average_rating"
    ] = summary[
        "average_rating"
    ].round(2)

    examples = subset[
        [
            "product_name",
            "reviews.rating",
            "reviews.text"
        ]
    ].dropna().head(10)

    return (
        summary,
        examples
    )

# Task 3 Results

## 10. Load Qwen recommendation articles

Task 3 generates `category_recommendation_articles.csv`.

The deployed app displays these generated results instead of loading the Qwen model for every web request. This makes deployment substantially lighter.

In [32]:
if not ARTICLES_PATH.exists():
    raise FileNotFoundError(
        "Task 3 output category_recommendation_articles.csv was not found."
    )

articles = pd.read_csv(
    ARTICLES_PATH
)

CATEGORY_ARTICLE_COLUMN = (
    "kmeans_category"
    if "kmeans_category" in articles.columns
    else "meta_category"
)

required_article_columns = [
    CATEGORY_ARTICLE_COLUMN,
    "generated_article"
]

missing = [
    col for col in required_article_columns
    if col not in articles.columns
]

if missing:
    raise ValueError(
        f"Task 3 output is missing: {missing}"
    )

articles = articles.dropna(
    subset=[
        CATEGORY_ARTICLE_COLUMN,
        "generated_article"
    ]
).copy()

print(
    "Generated articles:",
    len(articles)
)

display(
    articles[
        [
            CATEGORY_ARTICLE_COLUMN,
            "generated_article"
        ]
    ].head()
)

Generated articles: 6


,kmeans_category,generated_article
0,Fire Tablets,### Category Overview\nThe Fire Tablets market...
1,E-readers & Kindle,### Category Overview\nThe Kindle series has b...
2,Streaming & TV,### Category Overview\nThe streaming and TV ma...
3,Chargers & Cables,"### Category Overview\n\nThe category ""Charger..."
4,Smart Speakers & Echo,### Category Overview\nThe Smart Speakers & Ec...


## 11. Recommendation article function

In [33]:
ARTICLE_CHOICES = sorted(
    articles[
        CATEGORY_ARTICLE_COLUMN
    ]
    .astype(str)
    .unique()
    .tolist()
)

def get_recommendation_article(category):

    if not category:
        return "Select a category."

    match = articles[
        articles[
            CATEGORY_ARTICLE_COLUMN
        ].astype(str) == str(category)
    ]

    if match.empty:
        return (
            "No generated recommendation article "
            "is available for this category."
        )

    return str(
        match.iloc[0][
            "generated_article"
        ]
    )

# 12. Test the three components

In [34]:
print("TASK 1 TEST")
print(
    predict_sentiment(
        "This product works perfectly and I love it."
    )
)

print("\nTASK 2 CATEGORIES")
print(CATEGORY_CHOICES)

print("\nTASK 3 ARTICLES")
print(ARTICLE_CHOICES)

TASK 1 TEST
('Positive', {'Positive': 0.9971712231636047, 'Neutral': 0.002600087085738778, 'Negative': 0.00022866390645503998})

TASK 2 CATEGORIES
['Chargers & Cables', 'E-readers & Kindle', 'Fire Tablets', 'Household, Office & Pets', 'Smart Speakers & Echo', 'Streaming & TV']

TASK 3 ARTICLES
['Chargers & Cables', 'E-readers & Kindle', 'Fire Tablets', 'Household, Office & Pets', 'Smart Speakers & Echo', 'Streaming & TV']


# 13. Build the integrated Gradio application

The app uses three tabs corresponding to the three project tasks.

In [35]:
with gr.Blocks(
    title="Amazon Review NLP System"
) as demo:

    gr.Markdown(
        "# Amazon Product Review NLP System\n"
        "Sentiment classification, product-category "
        "exploration, and AI-generated recommendations."
    )

    with gr.Tab(
        "Sentiment Analyzer"
    ):

        gr.Markdown(
            "### Task 1 — RoBERTa Sentiment Classification"
        )

        review_input = gr.Textbox(
            lines=6,
            label="Amazon Product Review",
            placeholder=(
                "Enter a product review..."
            )
        )

        analyze_button = gr.Button(
            "Analyze Sentiment"
        )

        sentiment_output = gr.Textbox(
            label="Predicted Sentiment"
        )

        confidence_output = gr.Label(
            num_top_classes=3,
            label="Confidence Scores"
        )

        analyze_button.click(
            fn=predict_sentiment,
            inputs=review_input,
            outputs=[
                sentiment_output,
                confidence_output
            ]
        )

        gr.Examples(
            examples=[
                [
                    "This product is excellent and works perfectly."
                ],
                [
                    "The product is okay. Nothing special."
                ],
                [
                    "Terrible product. It stopped working after two days."
                ]
            ],
            inputs=review_input
        )

    with gr.Tab(
        "Category Explorer"
    ):

        gr.Markdown(
            "### Task 2 — Product Clustering"
        )

        gr.Dataframe(
            value=category_summary,
            label="Category Overview",
            interactive=False
        )

        category_input = gr.Dropdown(
            choices=CATEGORY_CHOICES,
            value=(
                CATEGORY_CHOICES[0]
                if CATEGORY_CHOICES
                else None
            ),
            label="Select Product Category"
        )

        explore_button = gr.Button(
            "Explore Category"
        )

        product_table = gr.Dataframe(
            label="Products in Category",
            interactive=False
        )

        review_table = gr.Dataframe(
            label="Example Reviews",
            interactive=False
        )

        explore_button.click(
            fn=explore_category,
            inputs=category_input,
            outputs=[
                product_table,
                review_table
            ]
        )

    with gr.Tab(
        "AI Recommendations"
    ):

        gr.Markdown(
            "### Task 3 — Qwen Recommendation Articles\n"
            "These articles were generated from "
            "Task 2 clusters and review evidence."
        )

        article_category = gr.Dropdown(
            choices=ARTICLE_CHOICES,
            value=(
                ARTICLE_CHOICES[0]
                if ARTICLE_CHOICES
                else None
            ),
            label="Select Category"
        )

        article_button = gr.Button(
            "Show Recommendation"
        )

        article_output = gr.Markdown()

        article_button.click(
            fn=get_recommendation_article,
            inputs=article_category,
            outputs=article_output
        )

    gr.Markdown(
        "---\n"
        "**Pipeline:** Reviews → RoBERTa sentiment "
        "classification → product clustering → "
        "Qwen recommendation synthesis."
    )

print("Integrated Gradio app created.")

Integrated Gradio app created.


## 14. Launch locally

In [36]:
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3dbc40c7a1ca74367b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Deployment Files

The next cells create a standalone `app.py` based on the same three-task integration.

For Hugging Face Spaces, upload:

```text
app.py
requirements.txt
models/amazon_roberta_sentiment_model/
datasets/clustered_reviews.csv
category_recommendation_articles.csv
```

## 15. Generate standalone `app.py`

In [37]:
app_source = r"""
from pathlib import Path
import pandas as pd
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = Path("model/amazon_roberta_sentiment_model")
CLUSTERED_PATH = Path("datasets/clustered_reviews.csv")
ARTICLES_PATH = Path("category_recommendation_articles.csv")

DEFAULT_LABELS = {0:"Negative", 1:"Neutral", 2:"Positive"}

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH))
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_PATH))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

stored = getattr(model.config, "id2label", {}) or {}
ID2LABEL = {}
for i in range(model.config.num_labels):
    label = str(stored.get(i, stored.get(str(i), "")))
    if not label or label.upper().startswith("LABEL_"):
        label = DEFAULT_LABELS.get(i, f"Class {i}")
    else:
        label = label.capitalize()
    ID2LABEL[i] = label

clustered = pd.read_csv(CLUSTERED_PATH, low_memory=False)
articles = pd.read_csv(ARTICLES_PATH)
article_category_col = (
    "kmeans_category"
    if "kmeans_category" in articles.columns
    else "meta_category"
)

category_summary = (
    clustered.groupby("kmeans_category")
    .agg(
        products=("product_name","nunique"),
        reviews=("reviews.text","count"),
        average_rating=("reviews.rating","mean")
    )
    .reset_index()
)
category_summary["average_rating"] = category_summary["average_rating"].round(2)

categories = sorted(clustered["kmeans_category"].dropna().unique().tolist())
article_categories = sorted(
    articles[article_category_col].dropna().astype(str).unique().tolist()
)

def predict_sentiment(review):
    if review is None or not review.strip():
        return "Please enter a review.", {}

    inputs = tokenizer(
        review, return_tensors="pt",
        truncation=True, max_length=256, padding=True
    )
    inputs = {k:v.to(device) for k,v in inputs.items()}

    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=-1)[0]

    idx = int(torch.argmax(probs).item())
    scores = {ID2LABEL[i]:float(probs[i]) for i in range(len(probs))}
    scores = dict(sorted(scores.items(), key=lambda x:x[1], reverse=True))
    return ID2LABEL[idx], scores

def explore_category(category):
    d = clustered[clustered["kmeans_category"] == category].copy()
    products = (
        d.groupby("product_name")
        .agg(
            review_count=("reviews.rating","count"),
            average_rating=("reviews.rating","mean")
        )
        .reset_index()
        .sort_values(["average_rating","review_count"], ascending=[False,False])
    )
    products["average_rating"] = products["average_rating"].round(2)
    examples = d[
        ["product_name","reviews.rating","reviews.text"]
    ].dropna().head(10)
    return products, examples

def get_article(category):
    match = articles[
        articles[article_category_col].astype(str) == str(category)
    ]
    if match.empty:
        return "No recommendation article is available."
    return str(match.iloc[0]["generated_article"])

with gr.Blocks(title="Amazon Review NLP System") as demo:
    gr.Markdown(
        "# Amazon Product Review NLP System\n"
        "Sentiment classification, category clustering, "
        "and AI-generated recommendations."
    )

    with gr.Tab("Sentiment Analyzer"):
        gr.Markdown("### Task 1 — RoBERTa")
        review = gr.Textbox(lines=6, label="Product Review")
        button = gr.Button("Analyze Sentiment")
        sentiment = gr.Textbox(label="Predicted Sentiment")
        confidence = gr.Label(num_top_classes=3, label="Confidence Scores")
        button.click(predict_sentiment, review, [sentiment, confidence])

    with gr.Tab("Category Explorer"):
        gr.Markdown("### Task 2 — K-Means Product Categories")
        gr.Dataframe(value=category_summary, interactive=False)
        category = gr.Dropdown(
            choices=categories,
            value=categories[0] if categories else None,
            label="Category"
        )
        explore = gr.Button("Explore Category")
        products = gr.Dataframe(label="Products", interactive=False)
        reviews = gr.Dataframe(label="Example Reviews", interactive=False)
        explore.click(explore_category, category, [products, reviews])

    with gr.Tab("AI Recommendations"):
        gr.Markdown("### Task 3 — Qwen Recommendation Articles")
        article_category = gr.Dropdown(
            choices=article_categories,
            value=article_categories[0] if article_categories else None,
            label="Category"
        )
        show = gr.Button("Show Recommendation")
        article = gr.Markdown()
        show.click(get_article, article_category, article)

if __name__ == "__main__":
    demo.launch()
"""

Path("app.py").write_text(
    app_source.strip() + "\n",
    encoding="utf-8"
)

print("Created app.py")

Created app.py


## 16. Generate `requirements.txt`

In [38]:
requirements = """gradio
pandas
torch
transformers
safetensors
"""

Path("requirements.txt").write_text(
    requirements,
    encoding="utf-8"
)

print("Created requirements.txt")

Created requirements.txt


## 17. Validate deployment inputs

In [39]:
deployment_paths = [
    Path("app.py"),
    Path("requirements.txt"),
    MODEL_PATH,
    CLUSTERED_REVIEWS_PATH,
    ARTICLES_PATH
]

for path in deployment_paths:
    print(
        "READY" if path.exists() else "MISSING",
        "->",
        path
    )

READY -> app.py
READY -> requirements.txt
READY -> model/amazon_roberta_sentiment_model
READY -> datasets/clustered_reviews.csv
READY -> category_recommendation_articles.csv


# 18. Hugging Face Spaces structure

Use this structure:

```text
Amazon-Review-NLP/
│
├── app.py
├── requirements.txt
├── category_recommendation_articles.csv
│
├── datasets/
│   └── clustered_reviews.csv
│
└── models/
    └── amazon_roberta_sentiment_model/
        ├── config.json
        ├── model.safetensors
        ├── tokenizer.json
        ├── tokenizer_config.json
        ├── vocab.json
        └── merges.txt
```

The exact tokenizer files can differ depending on the saved Task 1 model.

In [ ]:
from huggingface_hub import notebook_login, upload_file, upload_folder


import os
os.environ['HF_TOKEN'] = 'HF_TOKEN'

notebook_login()

upload_file(
    path_or_fileobj="app.py",
    path_in_repo="app.py",
    repo_id="vinaypyatimani/cuRev",
)

upload_file(
    path_or_fileobj="requirements.txt",
    path_in_repo="requirements.txt",
    repo_id="vinaypyatimani/cuRev",
)

upload_file(
    path_or_fileobj="datasets/clustered_reviews.csv",
    path_in_repo="datasets/clustered_reviews.csv",
    repo_id="vinaypyatimani/cuRev",
)

upload_folder(
    folder_path="model/amazon_roberta_sentiment_model",
    path_in_repo="models/amazon_roberta_sentiment_model",
    repo_id="vinaypyatimani/cuRev",
)

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ets/clustered_reviews.csv:  20%|##        | 7.99MB / 39.7MB            

Validating                    :           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/vinaypyatimani/CuRev/commit/c529d2346e0744cda7676cdda9bf7a51a9682c39', commit_message='Upload folder using huggingface_hub', commit_description='', oid='c529d2346e0744cda7676cdda9bf7a51a9682c39', pr_url=None, repo_url=RepoUrl('https://huggingface.co/vinaypyatimani/CuRev', endpoint='https://huggingface.co', repo_type='model', repo_id='vinaypyatimani/CuRev'), pr_revision=None, pr_num=None)

# 19. Final Project Flow

```text
                         ┌─────────────────────────────┐
                         │ Amazon Product Review App   │
                         └──────────────┬──────────────┘
                                        │
                ┌───────────────────────┼───────────────────────┐
                │                       │                       │
                ▼                       ▼                       ▼
       Sentiment Analyzer       Category Explorer       AI Recommendations
                │                       │                       │
         Task 1 RoBERTa           Task 2 K-Means          Task 3 Qwen
                │                       │                       │
        Negative/Neutral/       6 product meta-        grounded category
        Positive + scores       categories + stats     recommendation articles
```

This version demonstrates the complete project rather than deploying Task 1 in isolation.